# Unsafe Vehicle Load Detection with AI

This notebook demonstrates how to detect unsafe vehicle loads using deep learning.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
from model import get_model

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Setup and Configuration

In [ ]:
# Configuration
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 20
IMG_SIZE = 224
NUM_CLASSES = 2  # safe, unsafe

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Data Preparation

In [ ]:
# Create data directories
os.makedirs('data/train/safe', exist_ok=True)
os.makedirs('data/train/unsafe', exist_ok=True)
os.makedirs('data/test', exist_ok=True)
os.makedirs('models', exist_ok=True)

print('Data directories created!')
print('Place your images in:')
print('  - data/train/safe/ (for safe load images)')
print('  - data/train/unsafe/ (for unsafe load images)')

In [ ]:
# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

## 3. Model Architecture

In [ ]:
# Load model
model = get_model(num_classes=NUM_CLASSES, pretrained=True)
model = model.to(device)

print('Model loaded successfully!')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Training

In [ ]:
# Train the model
from train import train_model

model, history = train_model(
    data_dir='data/train',
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE
)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'])
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(history['accuracy'])
ax2.set_title('Training Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True)

plt.tight_layout()
plt.show()

## 5. Inference

In [ ]:
def predict_and_visualize(image_path, model_path='models/best_model.pth'):
    # Load model
    model = get_model(num_classes=2, pretrained=False)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = test_transform(image).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        predicted_class = outputs.argmax(1).item()
        confidence = probabilities[0][predicted_class].item()
    
    classes = ['Safe Load', 'Unsafe Load']
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.imshow(image)
    ax1.axis('off')
    ax1.set_title(f'Prediction: {classes[predicted_class]}\nConfidence: {confidence*100:.2f}%')
    
    probs = probabilities[0].cpu().numpy() * 100
    ax2.bar(classes, probs, color=['green', 'red'])
    ax2.set_ylabel('Probability (%)')
    ax2.set_title('Class Probabilities')
    ax2.set_ylim([0, 100])
    
    plt.tight_layout()
    plt.show()
    
    return classes[predicted_class], confidence

In [ ]:
# Test on a sample image
# Replace with your image path
test_image = 'data/test/sample.jpg'

if os.path.exists(test_image):
    prediction, confidence = predict_and_visualize(test_image)
    print(f'\nPrediction: {prediction}')
    print(f'Confidence: {confidence*100:.2f}%')
else:
    print(f'Please place a test image at: {test_image}')

## 6. Batch Prediction

In [ ]:
# Predict on multiple images
test_dir = 'data/test'
if os.path.exists(test_dir):
    test_images = [f for f in os.listdir(test_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    for img_name in test_images[:5]:  # Show first 5
        img_path = os.path.join(test_dir, img_name)
        print(f'\nProcessing: {img_name}')
        predict_and_visualize(img_path)
else:
    print('No test directory found')